# Energy Landscape 

## Loading Model

In [1]:
"""Bare-bones energy landscape example for V-JEPA 2-AC."""

import sys
from pathlib import Path

# REPO_DIR = Path(__file__).resolve().parent.parent
REPO_DIR = Path.cwd().parent 
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "notebooks"))

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.nn import functional as F
from app.vjepa_droid.transforms import make_transforms
import time

# ---- Device ----------------------------------------------------------------
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cuda"
dtype = torch.float32
print(f"Using device={device}, dtype={dtype}")

# ---- Model -----------------------------------------------------------------
print("Loading model...")
encoder, predictor = torch.hub.load(
    str(REPO_DIR), "vjepa2_ac_vit_giant", source="local", trust_repo=True,
)
print("Model loaded.")

Using device=cuda, dtype=torch.float32
Loading model...


/home/hashim/miniconda3/envs/vjepa2_env/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Model loaded.


## Model 

In [2]:
from utils.world_model_wrapper import WorldModel
from utils.mpc_utils import compute_new_pose, poses_to_diff

encoder.eval().to(device=device, dtype=dtype)
predictor.eval().to(device=device, dtype=dtype)

crop_size = 256
tokens_per_frame = (crop_size // encoder.patch_size) ** 2
transform = make_transforms(
    random_horizontal_flip=False,
    random_resize_aspect_ratio=(1., 1.),
    random_resize_scale=(1., 1.),
    reprob=0., auto_augment=False, motion_shift=False,
    crop_size=crop_size,
)

In [3]:
# ---- Trajectory ------------------------------------------------------------
play_in_reverse = False
traj = np.load(Path.cwd() / "franka_example_traj.npz")
np_clips = traj["observations"]
np_states = traj["states"]
if play_in_reverse:
    np_clips = np_clips[:, ::-1].copy()
    np_states = np_states[:, ::-1].copy()
np_actions = np.expand_dims(poses_to_diff(np_states[0, 0], np_states[0, 1]), axis=(0, 1))

clips = transform(np_clips[0]).unsqueeze(0).to(device=device, dtype=dtype)
states = torch.tensor(np_states).to(device=device, dtype=dtype)
gt_actions = torch.tensor(np_actions).to(device=device, dtype=dtype)


In [7]:
print(f"Total frames available: {np_clips.shape[1]}")

Total frames available: 2


## Helpers

In [ ]:
# ---- Forward helpers -------------------------------------------------------
def forward_target(c, normalize=True):
    B, C, T, H, W = c.size()
    c = c.permute(0, 2, 1, 3, 4).flatten(0, 1).unsqueeze(2).repeat(1, 1, 2, 1, 1)
    h = encoder(c)
    h = h.view(B, T, -1, h.size(-1)).flatten(1, 2)
    return F.layer_norm(h, (h.size(-1),)) if normalize else h

def energy(z, h):
    return torch.abs(z[:, -tokens_per_frame:] - h[:, -tokens_per_frame:]).mean(dim=[1, 2]).tolist()

## Prediction

In [5]:
# ---- Predict optimal action via gradient-based MPC -------------------------
ground_truth_action = gt_actions[0, 0].tolist()
noise = np.random.uniform(-0.03, 0.03, size=7).tolist()
warmstart = np.array([gt + n for gt, n in zip(ground_truth_action, noise)])

world_model = WorldModel(
    encoder=encoder,
    predictor=predictor,
    tokens_per_frame=tokens_per_frame,
    transform=transform,
    mpc_args={
        "rollout": 1,
        "warmstart": warmstart,
        "maxnorm": 0.15,
    },
    normalize_reps=True,
    device=device,
)

start_time = time.time()
with torch.no_grad():
    h = forward_target(clips)
z_n, z_goal = h[:, :tokens_per_frame], h[:, -tokens_per_frame:]
print("Starting planning using Gradient Descent...")
pred = world_model.infer_next_action_gradient(z_n, states[:, :1], z_goal).cpu().numpy()
end_time = time.time()

gt = gt_actions[0, 0].tolist()
print(f"Planning completed in {end_time - start_time:.2f} seconds.")
print(f"Warmstart action (x,y,z) = ({warmstart[0]:.3f},{warmstart[1]:.3f},{warmstart[2]:.3f})")
print(f"Gradient Descent predicted action (x,y,z) = ({pred[0, 0]:.3f},{pred[0, 1]:.3f},{pred[0, 2]:.3f})")
print(f"Ground truth action (x,y,z) = ({gt[0]:.3f},{gt[1]:.3f},{gt[2]:.3f})")

/home/hashim/miniconda3/envs/vjepa2_env/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Starting planning using Gradient Descent...
[gradient] step 0000 total=0.428125 latent=0.428125 mag=0.000000 prior=0.000000 action_xyz=(+0.0736, +0.0391, +0.1062)
[gradient] step 0010 total=0.422133 latent=0.421099 mag=0.000000 prior=0.000103 action_xyz=(+0.0836, +0.0512, +0.0982)
[gradient] step 0020 total=0.419432 latent=0.416072 mag=0.000000 prior=0.000336 action_xyz=(+0.0904, +0.0603, +0.0896)
[gradient] step 0030 total=0.419249 latent=0.414610 mag=0.000000 prior=0.000464 action_xyz=(+0.0899, +0.0638, +0.0835)
[gradient] step 0040 total=0.418862 latent=0.414453 mag=0.000000 prior=0.000441 action_xyz=(+0.0849, +0.0645, +0.0828)
[gradient] step 0050 total=0.418749 latent=0.414246 mag=0.000000 prior=0.000450 action_xyz=(+0.0816, +0.0678, +0.0846)
[gradient] step 0060 total=0.418758 latent=0.413973 mag=0.000000 prior=0.000478 action_xyz=(+0.0815, +0.0691, +0.0844)
[gradient] step 0070 total=0.418739 latent=0.414139 mag=0.000000 prior=0.000460 action_xyz=(+0.0820, +0.0674, +0.0836)
[gra